In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from estimation_forecast_functions_var import DataCleaner_SGD
import holidays
from simple_strat_funcs import Clean_Implied_Vols_SGD_with_smile
from scipy.stats import norm
from hedging_strategy_class_NEW_KURT_SKEW_vega_VAR import Compare_Trading_Strategies
import seaborn as sns

/Users/alexvillamartin/Documents/MSc Diss/Code/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


This one used log returns for sharpe and uses the domestic rate to compute risk free returns not SOFR like we did prior.

# Global params

In [2]:
train_size = 0.7
test_size = 1 - train_size

TICKER = "USDSGD"

# always use same amount of money in USD to start and convert where necessary
IC_USD = 1_000_000 # initial capital in USD
current_rate = 1.28
IC_SGD = IC_USD * current_rate
NOTIONAL_USD = 1_000_000 # this is used in strategy as our notional to ensure trade positions are in domestic currency

MAX_DELTA_DIFF = 500 # this param doesnt change much as our delta positions are usually much larger
SIGNAL_LB = 0.01
SIGNAL_UB = 0.99
TRANSACTION_COST_BOOL = True
TRANSACTION_COSTS_SPOT = 0.0002 / 2 # for half a leg ie selling/buying once not to enter and exit the trade
TRANSACTION_COSTS_OPTION = 0.0005 / 2 # [change to vol spreads later] for half a leg again, both in decimals

HYPERPARAM_SORT = 'sharpe_ratio' # can be 'sharpe_ratio', 'cagr', 'max_drawdown', 'total_return'
GARCH_1_2_INDICATOR = False

k_bar_MSM = 8
b_MSM = 2.0
gamma_kbar_MSM = 0.5

M = 300 # figarch lags

CONVERT_USD_INDICATOR = True # convert all portfolio values and metrics to USD for fair comparison - use when USD is not domestic

long_threshs = np.array([1.02, 1.03, 1.04, 1.05, 1.06, 1.07,  1.08, 1.09, 1.10, 1.11, 1.12, 1.13, 1.14, 1.15, 1.16, 1.17, 1.18, 1.19, 1.20, 1.21, 1.22, 1.23, 1.24, 1.25, 1.26, 1.27])
short_threshs = np.array([0.98, 0.97, 0.96, 0.95, 0.94, 0.93, 0.92,  0.91, 0.90, 0.89, 0.88, 0.87, 0.86, 0.85, 0.84, 0.83, 0.82, 0.81, 0.80, 0.79, 0.78, 0.77, 0.76, 0.75, 0.74, 0.73])
sig_multipliers = np.array([1, 3, 5, 7, 9, 11, 13, 15, 17, 20])

# Global data

In [3]:
usdsgd_5m = pd.read_parquet("SGD_USD_5M.parquet").copy()
usdsgd_5m.rename(columns={'c': 'spot'}, inplace=True)  
cleaner_usdsgd = DataCleaner_SGD(usdsgd_5m, start_hr=2, end_hr=18, unit_test=False) 
realised_variance_usdsgd = cleaner_usdsgd.clean_data()
daily_log_returns_usdsgd = pd.read_parquet("DF_D_USDSGD.parquet")['Log_r']
start_date = pd.to_datetime(realised_variance_usdsgd.index.min())
end_date = pd.to_datetime(realised_variance_usdsgd.index.max())
spot_curr = pd.read_parquet("/Users/alexvillamartin/Documents/MSc Diss/Code/DF_D_USDSGD.parquet")
overnight_domestic_rate = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/SORA_daily.csv").set_index("Date") 
overnight_domestic_rate.index = (pd.to_datetime(overnight_domestic_rate.index, format='mixed').normalize())
overnight_foreign_rate = pd.read_csv("SOFR_daily.csv").set_index("date")

# Global functions

Change holidays where needed. 

In [4]:
def align_spots(df, aligned_df, start, end):

    df = df[(df.index >= start) & (df.index <= end)]

    years = range(start.year, end.year + 1)

    first = holidays.US(years=years)
    second = holidays.Singapore(years=years)

    hols = pd.to_datetime(list(set(first) | set(second))).normalize()

    idx  = df.index
    mask = ~idx.isin(hols)
    new_df = df.loc[mask]

    df_aligned_dates = aligned_df.index
    common_dates = new_df.index.intersection(df_aligned_dates)
    new_df = new_df.loc[common_dates]

    return new_df

def sort_rates(r_b, r_t, overn_r, aligned_df, overn_for_r):

    r_b.index = pd.to_datetime(r_b.index)
    valid_mask = ~( 
                   r_t.index.isna())
    r_t_clean = r_t[valid_mask]
    r_t_clean.index = pd.to_datetime(r_t_clean.index, format='%Y-%m-%d')
    
    valid_mask2 = ~(
                   overn_r.index.isna())
    overn_r_clean = overn_r[valid_mask2]
    overn_r_clean.index = pd.to_datetime(overn_r_clean.index, format='%Y-%m-%d')

    valid_mask3 = ~(
                   overn_for_r.index.isna())
    overn_for_r_clean = overn_for_r[valid_mask3]
    overn_for_r_clean.index = pd.to_datetime(overn_for_r_clean.index, format='mixed').normalize()    

    r_b = r_b.reindex(aligned_df.index)
    r_t_clean = r_t_clean.reindex(aligned_df.index)
    overn_r_clean = overn_r_clean.reindex(aligned_df.index)
    overn_for_r_clean = overn_for_r_clean.reindex(aligned_df.index)

    r_b = r_b.ffill()
    r_t_clean = r_t_clean.ffill()
    overn_r_clean = overn_r_clean.ffill()
    overn_for_r_clean = overn_for_r_clean.ffill()

    # convert into decimals and continously compunded versions for BSE
    r_b = pd.to_numeric(r_b['rate_pct'], errors='coerce')
    r_t_clean = pd.to_numeric(r_t_clean['rate_pct'], errors='coerce')
    overn_r_clean = pd.to_numeric(overn_r_clean['SORA'], errors='coerce')
    r_b = r_b / 100
    r_t_clean = r_t_clean / 100
    overn_r_clean = overn_r_clean / 100 # not continously compounded
    overn_r_clean *= 1/365 # daily

    overn_for_r_clean = pd.to_numeric(overn_for_r_clean['value'], errors='coerce')
    overn_for_r_clean = overn_for_r_clean / 100 # not continously compounded
    overn_for_r_clean *= 1/365 # daily

    r_b = np.log(1 + r_b)
    r_t_clean = np.log(1 + r_t_clean)

    return r_b, r_t_clean, overn_r_clean, overn_for_r_clean

# H=30, T=1Mo

In [5]:
OPTION_MATURITY_1 = 1 / 12
H_1 = 30 

implied_vol_data_1 = pd.read_excel('/Users/alexvillamartin/Documents/MSc Diss/Code/USDSGD_1MO_ATM_D.xlsx')
vol_smile_data_1 = pd.read_csv("usdsgd_vol_smile_1mo_extra.csv").set_index("CalculationDate")
Data_clean_1 = Clean_Implied_Vols_SGD_with_smile(data=implied_vol_data_1, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_usdsgd, 
                                align_df2=daily_log_returns_usdsgd, 
                                smile_df=vol_smile_data_1)
implied_vol_data_1, realised_variance_1, daily_log_returns_1, vol_smile_data_1 = Data_clean_1.get_clean_data()

N_1 = len(daily_log_returns_1)
test_align_1 = daily_log_returns_1.iloc[N_1//2:-H_1]
spot_curr_test_1 = align_spots(spot_curr, test_align_1, start_date, end_date)

r_b_1 = pd.read_csv("SOFR_1mo_compounded.csv").set_index("date")
r_t_1 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/SORA_1mo_compounded.csv").set_index("Date")[['rate_pct']]

r_b_test_1, r_t_test_1, overn_dom_r_test_1, overn_for_r_test_1= sort_rates(r_b_1, r_t_1, overnight_domestic_rate, test_align_1, overnight_foreign_rate) 
r_b_test_1.ffill(inplace=True)

In [6]:
strategy_1 = Compare_Trading_Strategies(
    return_series=daily_log_returns_1, 
    realised_variance_series=realised_variance_1,
    atm_implied_vol_data=implied_vol_data_1,
    vol_smile_data=vol_smile_data_1,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_SGD,
    notional_base=NOTIONAL_USD,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_1,
    forecast_horizon=H_1,
    spot_series=spot_curr_test_1,
    overnight_domestic_rate=overn_dom_r_test_1,
    overnight_foreign_rate=overn_for_r_test_1,
    domestic_rate=r_t_test_1,
    foreign_rate=r_b_test_1, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers)

strategy_1.prepare_universal_series()
strategy_1.get_BMSM_data()
strategy_1.get_GARCH_data()
strategy_1.get_FIGARCH_data()

Estimated parameters: m0=1.245665e+00, sigma_bar=3.131619e-01
Final log-likelihood: -2.454608e+02
Estimated parameters: m0=1.219485e+00, sigma_bar=3.034983e-01
Final log-likelihood: -3.488166e+02
Estimated parameters: omega=0.0005432928150611904, alpha=0.0506, beta=0.9455
Estimated parameters: omega=0.0006403105521506784, alpha=0.0481, beta=0.9463
Estimated parameters: omega=0.015803365104575497, d=0.3245, beta=0.2874
Final log-likelihood = 1629.6295
Estimated parameters: omega=0.016044982742265446, d=0.2844, beta=0.2535
Final log-likelihood = 2907.9452


In [7]:
error_metrics_df_1, log_ls, m_z_results, se_results = strategy_1.in_sample_predictions()

In [8]:
error_metrics_df_1

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,0.658040,NaN,NaN,0.700758,NaN,NaN
BMSM OLS,0.659361,0.016724,0.506670,0.739561,1.149219,0.874649
GARCH,0.621147,NaN,NaN,0.644777,NaN,NaN
GARCH OLS,0.564845,-1.064922,0.143567,0.646645,0.074323,0.529617
FIGARCH,0.651932,NaN,NaN,0.699007,NaN,NaN
FIGARCH OLS,0.544045,-1.616522,0.053127,0.622852,-2.147871,0.015965


In [11]:
log_ls

{'BMSM': np.float64(-245.46079263864053),
 'GARCH': np.float64(1669.5132899315033),
 'FIGARCH': np.float64(1629.6295201908133)}

In [9]:
m_z_results

{'BMSM': {'alpha_hat': -0.0004853240314810711,
  'beta_hat': 1.2624788858145317,
  'alpha_p': 0.07163879627989612,
  'beta_p': 0.05068249837259231},
 'GARCH': {'alpha_hat': 0.0006137042815081922,
  'beta_hat': 0.7966799834889942,
  'alpha_p': 4.180222733375696e-05,
  'beta_p': 0.01071277526833223},
 'FIGARCH': {'alpha_hat': -0.00029888198278982497,
  'beta_hat': 1.0920056211700278,
  'alpha_p': 0.2727063255103187,
  'beta_p': 0.4478654961938052}}

In [12]:
se_results

{'BMSM': array([0.02186719, 0.0222432 ]),
 'GARCH': array([0.0010078 , 0.00666329, 0.00725365]),
 'FIGARCH': array([0.00249443, 0.04306692, 0.04883622])}

In [10]:
strategy_1.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.115
Model:                            OLS   Adj. R-squared:                  0.111
Method:                 Least Squares   F-statistic:                     8.297
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           1.34e-06
Time:                        22:23:11   Log-Likelihood:                -712.78
No. Observations:                1158   AIC:                             1436.
Df Residuals:                    1153   BIC:                             1461.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0818      0.033     -2.445      0.014      -0.147      -0.016
x1             0.2242      0.062      3.595      0.000       0.102       0.346
x2            -0.1411      0.079     -1.794      0.073      -0.295       0.013
x3             0.0804      0.043      1.878      0.060      -0.004       0.164
x4            -0.0043      0.050     -0.085      0.932      -0.103       0.095
==============================================================================
Omnibus:                       96.178   Durbin-Watson:                   0.060
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              118.747
Skew:                           0.768   Prob(JB):                     1.64e-26
Kurtosis:                       3.315   Cond. No.                         6.01
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [13]:
strategy_1.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.052
Model:                            OLS   Adj. R-squared:                  0.049
Method:                 Least Squares   F-statistic:                     5.324
Date:                Sat, 30 Aug 2025   Prob (F-statistic):            0.00121
Time:                        22:23:34   Log-Likelihood:                -701.15
No. Observations:                1158   AIC:                             1410.
Df Residuals:                    1154   BIC:                             1431.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0541      0.034     -1.612      0.107      -0.120       0.012
x1            -0.0943      0.026     -3.691      0.000      -0.144      -0.044
x2             0.0680      0.042      1.637      0.102      -0.013       0.149
x3             0.0238      0.047      0.502      0.616      -0.069       0.117
==============================================================================
Omnibus:                       96.591   Durbin-Watson:                   0.044
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              119.512
Skew:                           0.775   Prob(JB):                     1.12e-26
Kurtosis:                       3.268   Cond. No.                         1.78
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [14]:
strategy_1.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.106
Model:                            OLS   Adj. R-squared:                  0.104
Method:                 Least Squares   F-statistic:                     5.092
Date:                Sat, 30 Aug 2025   Prob (F-statistic):            0.00167
Time:                        22:23:37   Log-Likelihood:                -822.57
No. Observations:                1158   AIC:                             1653.
Df Residuals:                    1154   BIC:                             1673.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1647      0.037     -4.439      0.000      -0.237      -0.092
x1            -0.0463      0.036     -1.287      0.198      -0.117       0.024
x2             0.1043      0.043      2.416      0.016       0.020       0.189
x3            -0.1161      0.053     -2.176      0.030      -0.221      -0.012
==============================================================================
Omnibus:                      101.512   Durbin-Watson:                   0.050
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              127.984
Skew:                           0.812   Prob(JB):                     1.62e-28
Kurtosis:                       3.110   Cond. No.                         1.90
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

# H=91, T=3MO

In [15]:
OPTION_MATURITY_2 = 3 / 12
H_2 = 91 

vol_smile_data_2 = pd.read_csv("usdsgd_vol_smile_3mo_extra.csv").set_index("CalculationDate")
implied_vol_data_2 = pd.DataFrame({
    'Exchange Date': vol_smile_data_2.index, 
    "Bid": vol_smile_data_2['ATM'], 
    "Ask": vol_smile_data_2['ATM'],
    "BidNet": vol_smile_data_2['ATM']})
Data_clean_2 = Clean_Implied_Vols_SGD_with_smile(data=implied_vol_data_2, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_usdsgd, 
                                align_df2=daily_log_returns_usdsgd, 
                                smile_df=vol_smile_data_2)
implied_vol_data_2, realised_variance_2, daily_log_returns_2, vol_smile_data_2 = Data_clean_2.get_clean_data()

N_2 = len(daily_log_returns_2)
test_align_2 = daily_log_returns_2.iloc[N_2//2:-H_2]
spot_curr_test_2 = align_spots(spot_curr, test_align_2, start_date, end_date)

r_b_2 = pd.read_csv("SOFR_3mo_compounded.csv").set_index("date")
r_t_2 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/SORA_3mo_compounded.csv").set_index("Date")[['rate_pct']]

r_b_test_2, r_t_test_2, overn_dom_r_test_2, overn_for_r_test_2= sort_rates(r_b_2, r_t_2, overnight_domestic_rate, test_align_2, overnight_foreign_rate) 
r_b_test_2.ffill(inplace=True)

In [16]:
strategy_2 = Compare_Trading_Strategies(
    return_series=daily_log_returns_2, 
    realised_variance_series=realised_variance_2,
    atm_implied_vol_data=implied_vol_data_2,
    vol_smile_data=vol_smile_data_2,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_SGD,
    notional_base=NOTIONAL_USD,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_2,
    forecast_horizon=H_2,
    spot_series=spot_curr_test_2,
    overnight_domestic_rate=overn_dom_r_test_2,
    overnight_foreign_rate=overn_for_r_test_2,
    domestic_rate=r_t_test_2,
    foreign_rate=r_b_test_2, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers)

strategy_2.prepare_universal_series()
strategy_2.get_BMSM_data()
strategy_2.get_GARCH_data()
strategy_2.get_FIGARCH_data()

Estimated parameters: m0=1.247284e+00, sigma_bar=3.110788e-01
Final log-likelihood: -2.421283e+02
Estimated parameters: m0=1.220630e+00, sigma_bar=3.026081e-01
Final log-likelihood: -3.441228e+02
Estimated parameters: omega=0.0005422359622443603, alpha=0.0509, beta=0.9452
Estimated parameters: omega=0.0006278439546422016, alpha=0.0483, beta=0.9462
Estimated parameters: omega=0.01602850765117938, d=0.3177, beta=0.2779
Final log-likelihood = 1668.1033
Estimated parameters: omega=0.016285062808385768, d=0.2822, beta=0.2479
Final log-likelihood = 2896.1939


In [17]:
error_metrics_df_2, _, m_z_results_2, _ = strategy_2.in_sample_predictions()

In [18]:
error_metrics_df_2

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,0.782777,NaN,NaN,0.830460,NaN,NaN
BMSM OLS,1.214635,2.577957,0.994967,1.069457,2.916061,0.998192
GARCH,0.770356,NaN,NaN,0.742755,NaN,NaN
GARCH OLS,0.877863,1.438646,0.924734,0.815843,1.642068,0.949571
FIGARCH,0.825716,NaN,NaN,0.892203,NaN,NaN
FIGARCH OLS,0.777965,-0.319344,0.374763,0.739994,-1.540467,0.061865


In [19]:
m_z_results_2

{'BMSM': {'alpha_hat': -0.0005283960490538396,
  'beta_hat': 1.2635694650621048,
  'alpha_p': 0.1557990349576207,
  'beta_p': 0.13811407265548536},
 'GARCH': {'alpha_hat': 0.0009067277475683525,
  'beta_hat': 0.630195984444076,
  'alpha_p': 7.591454688520229e-10,
  'beta_p': 4.139730597671124e-07},
 'FIGARCH': {'alpha_hat': -4.715621964118753e-05,
  'beta_hat': 0.9415239631004337,
  'alpha_p': 0.8854701512954014,
  'beta_p': 0.6671255663074641}}

In [20]:
strategy_2.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.228
Model:                            OLS   Adj. R-squared:                  0.225
Method:                 Least Squares   F-statistic:                     20.49
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           2.64e-16
Time:                        22:27:17   Log-Likelihood:                -556.68
No. Observations:                1116   AIC:                             1123.
Df Residuals:                    1111   BIC:                             1148.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1001      0.031     -3.269      0.001      -0.160      -0.040
x1             0.2329      0.069      3.362      0.001       0.097       0.369
x2             0.1113      0.067      1.650      0.099      -0.021       0.244
x3             0.0121      0.037      0.325      0.745      -0.061       0.085
x4             0.2415      0.065      3.720      0.000       0.114       0.369
==============================================================================
Omnibus:                       38.836   Durbin-Watson:                   0.066
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               42.168
Skew:                           0.470   Prob(JB):                     6.97e-10
Kurtosis:                       2.846   Cond. No.                         7.73
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [21]:
strategy_2.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.085
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                     2.647
Date:                Sat, 30 Aug 2025   Prob (F-statistic):             0.0478
Time:                        22:27:30   Log-Likelihood:                -593.11
No. Observations:                1116   AIC:                             1194.
Df Residuals:                    1112   BIC:                             1214.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1229      0.032     -3.850      0.000      -0.186      -0.060
x1             0.0037      0.030      0.125      0.901      -0.055       0.062
x2             0.0506      0.037      1.353      0.176      -0.023       0.124
x3             0.1571      0.061      2.570      0.010       0.037       0.277
==============================================================================
Omnibus:                       38.637   Durbin-Watson:                   0.031
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               35.948
Skew:                           0.388   Prob(JB):                     1.56e-08
Kurtosis:                       2.588   Cond. No.                         2.63
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [22]:
strategy_2.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.024
Model:                            OLS   Adj. R-squared:                  0.022
Method:                 Least Squares   F-statistic:                     1.755
Date:                Sat, 30 Aug 2025   Prob (F-statistic):              0.154
Time:                        22:27:34   Log-Likelihood:                -771.96
No. Observations:                1116   AIC:                             1552.
Df Residuals:                    1112   BIC:                             1572.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.2162      0.038     -5.727      0.000      -0.290      -0.142
x1            -0.0079      0.047     -0.169      0.865      -0.100       0.084
x2             0.0705      0.044      1.621      0.105      -0.015       0.156
x3            -0.0125      0.073     -0.170      0.865      -0.156       0.131
==============================================================================
Omnibus:                       61.355   Durbin-Watson:                   0.009
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               70.814
Skew:                           0.615   Prob(JB):                     4.20e-16
Kurtosis:                       2.901   Cond. No.                         2.65
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

# H=182, T=6MO

In [23]:
OPTION_MATURITY_3 = 6 / 12
H_3 = 182

vol_smile_data_3 = pd.read_csv("usdsgd_vol_smile_6mo_extra.csv").set_index("CalculationDate")
implied_vol_data_3 = pd.DataFrame({
    'Exchange Date': vol_smile_data_3.index, 
    "Bid": vol_smile_data_3['ATM'], 
    "Ask": vol_smile_data_3['ATM'],
    "BidNet": vol_smile_data_3['ATM']})
Data_clean_3 = Clean_Implied_Vols_SGD_with_smile(data=implied_vol_data_3, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_usdsgd, 
                                align_df2=daily_log_returns_usdsgd, 
                                smile_df=vol_smile_data_3)
implied_vol_data_3, realised_variance_3, daily_log_returns_3, vol_smile_data_3 = Data_clean_3.get_clean_data()

N_3 = len(daily_log_returns_3)
test_align_3 = daily_log_returns_3.iloc[N_3//2:-H_3]
spot_curr_test_3 = align_spots(spot_curr, test_align_3, start_date, end_date)

r_b_3 = pd.read_csv("SOFR_6mo_compounded.csv").set_index("date")
r_t_3 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/SORA_6mo_compounded.csv").set_index("Date")[['rate_pct']]

r_b_test_3, r_t_test_3, overn_dom_r_test_3, overn_for_r_test_3= sort_rates(r_b_3, r_t_3, overnight_domestic_rate, test_align_3, overnight_foreign_rate) 
r_b_test_3.ffill(inplace=True)

In [24]:
strategy_3 = Compare_Trading_Strategies(
    return_series=daily_log_returns_3, 
    realised_variance_series=realised_variance_3,
    atm_implied_vol_data=implied_vol_data_3,
    vol_smile_data=vol_smile_data_3,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_SGD,
    notional_base=NOTIONAL_USD,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_3,
    forecast_horizon=H_3,
    spot_series=spot_curr_test_3,
    overnight_domestic_rate=overn_dom_r_test_3,
    overnight_foreign_rate=overn_for_r_test_3,
    domestic_rate=r_t_test_3,
    foreign_rate=r_b_test_3, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers)

strategy_3.prepare_universal_series()
strategy_3.get_BMSM_data()
strategy_3.get_GARCH_data()
strategy_3.get_FIGARCH_data()

Estimated parameters: m0=1.247284e+00, sigma_bar=3.110788e-01
Final log-likelihood: -2.421283e+02
Estimated parameters: m0=1.222375e+00, sigma_bar=3.038406e-01
Final log-likelihood: -3.358341e+02
Estimated parameters: omega=0.0005422359622443603, alpha=0.0509, beta=0.9452
Estimated parameters: omega=0.0006166395366392186, alpha=0.0487, beta=0.9461
Estimated parameters: omega=0.01602850765117938, d=0.3177, beta=0.2779
Final log-likelihood = 1668.1033
Estimated parameters: omega=0.01609842452982523, d=0.2874, beta=0.2550
Final log-likelihood = 2795.5890


In [25]:
error_metrics_df_3, _, m_z_results_3, _ = strategy_3.in_sample_predictions()

In [26]:
error_metrics_df_3

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,0.997458,NaN,NaN,0.977057,NaN,NaN
BMSM OLS,1.615985,1.369252,0.914390,1.111695,0.671550,0.748989
GARCH,1.086141,NaN,NaN,0.930449,NaN,NaN
GARCH OLS,1.228659,0.285601,0.612379,0.890249,-0.185754,0.426337
FIGARCH,1.141623,NaN,NaN,1.080197,NaN,NaN
FIGARCH OLS,1.298698,0.214091,0.584741,0.853447,-0.618244,0.268276


In [27]:
m_z_results_3

{'BMSM': {'alpha_hat': 0.0012736727036875676,
  'beta_hat': 0.46440421743378485,
  'alpha_p': 0.007548790981869193,
  'beta_p': 0.009252873301609687},
 'GARCH': {'alpha_hat': 0.0015534223752501123,
  'beta_hat': 0.32625365587227334,
  'alpha_p': 1.7629102568010463e-14,
  'beta_p': 1.1489641844479455e-15},
 'FIGARCH': {'alpha_hat': 0.0020294590359867036,
  'beta_hat': 0.12081112879683797,
  'alpha_p': 9.305773311614887e-07,
  'beta_p': 3.2797507388829244e-09}}

In [28]:
strategy_3.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.350
Model:                            OLS   Adj. R-squared:                  0.347
Method:                 Least Squares   F-statistic:                     27.42
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           1.21e-21
Time:                        22:31:20   Log-Likelihood:                -355.17
No. Observations:                1025   AIC:                             720.3
Df Residuals:                    1020   BIC:                             745.0
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1373      0.028     -4.970      0.000      -0.192      -0.083
x1             0.2351      0.060      3.903      0.000       0.117       0.353
x2             0.0448      0.056      0.808      0.419      -0.064       0.154
x3             0.1117      0.037      3.058      0.002       0.040       0.183
x4             0.1425      0.042      3.377      0.001       0.060       0.225
==============================================================================
Omnibus:                       29.740   Durbin-Watson:                   0.046
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               28.008
Skew:                           0.358   Prob(JB):                     8.28e-07
Kurtosis:                       2.623   Cond. No.                         7.35
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [29]:
strategy_3.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.073
Model:                            OLS   Adj. R-squared:                  0.071
Method:                 Least Squares   F-statistic:                     6.194
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           0.000360
Time:                        22:31:24   Log-Likelihood:                -344.56
No. Observations:                1025   AIC:                             697.1
Df Residuals:                    1021   BIC:                             716.9
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.2496      0.027     -9.080      0.000      -0.303      -0.196
x1            -0.0028      0.027     -0.105      0.916      -0.055       0.050
x2             0.1439      0.034      4.172      0.000       0.076       0.211
x3             0.0849      0.036      2.332      0.020       0.014       0.156
==============================================================================
Omnibus:                       60.394   Durbin-Watson:                   0.029
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               26.249
Skew:                           0.160   Prob(JB):                     2.00e-06
Kurtosis:                       2.284   Cond. No.                         3.13
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [30]:
strategy_3.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.179
Model:                            OLS   Adj. R-squared:                  0.177
Method:                 Least Squares   F-statistic:                     10.95
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           4.43e-07
Time:                        22:31:27   Log-Likelihood:                -540.88
No. Observations:                1025   AIC:                             1090.
Df Residuals:                    1021   BIC:                             1109.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.2878      0.033     -8.602      0.000      -0.353      -0.222
x1            -0.0429      0.039     -1.104      0.270      -0.119       0.033
x2             0.1470      0.042      3.539      0.000       0.066       0.228
x3            -0.0712      0.043     -1.662      0.096      -0.155       0.013
==============================================================================
Omnibus:                       74.454   Durbin-Watson:                   0.023
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               34.074
Skew:                           0.247   Prob(JB):                     3.99e-08
Kurtosis:                       2.255   Cond. No.                         3.04
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

# H=364, T=1Y

In [31]:
OPTION_MATURITY_4 = 1
H_4 = 364

vol_smile_data_4 = pd.read_csv("usdsgd_vol_smile_1y_extra.csv").set_index("CalculationDate")
implied_vol_data_4 = pd.DataFrame({
    'Exchange Date': vol_smile_data_4.index, 
    "Bid": vol_smile_data_4['ATM'], 
    "Ask": vol_smile_data_4['ATM'],
    "BidNet": vol_smile_data_4['ATM']})
Data_clean_4 = Clean_Implied_Vols_SGD_with_smile(data=implied_vol_data_4, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_usdsgd, 
                                align_df2=daily_log_returns_usdsgd, 
                                smile_df=vol_smile_data_4)
implied_vol_data_4, realised_variance_4, daily_log_returns_4, vol_smile_data_4 = Data_clean_4.get_clean_data()

N_4 = len(daily_log_returns_4)
test_align_4 = daily_log_returns_4.iloc[N_4//2:-H_4]
spot_curr_test_4 = align_spots(spot_curr, test_align_4, start_date, end_date)

r_b_4 = pd.read_csv("SOFR_1y_compounded.csv").set_index("date")
r_t_4 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/SORA_1y_compounded.csv").set_index("Date")[['rate_pct']]

r_b_test_4, r_t_test_4, overn_dom_r_test_4, overn_for_r_test_4= sort_rates(r_b_4, r_t_4, overnight_domestic_rate, test_align_4, overnight_foreign_rate) 
r_b_test_4.ffill(inplace=True)

In [33]:
strategy_4 = Compare_Trading_Strategies(
    return_series=daily_log_returns_4, 
    realised_variance_series=realised_variance_4,
    atm_implied_vol_data=implied_vol_data_4,
    vol_smile_data=vol_smile_data_4,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_SGD,
    notional_base=NOTIONAL_USD,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_4,
    forecast_horizon=H_4,
    spot_series=spot_curr_test_4,
    overnight_domestic_rate=overn_dom_r_test_4,
    overnight_foreign_rate=overn_for_r_test_4,
    domestic_rate=r_t_test_4,
    foreign_rate=r_b_test_4, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers)

strategy_4.prepare_universal_series()
strategy_4.get_BMSM_data()
strategy_4.get_GARCH_data()
strategy_4.get_FIGARCH_data()

Estimated parameters: m0=1.247284e+00, sigma_bar=3.110788e-01
Final log-likelihood: -2.421283e+02
Estimated parameters: m0=1.221646e+00, sigma_bar=3.022316e-01
Final log-likelihood: -2.856906e+02
Estimated parameters: omega=0.0005422359622443603, alpha=0.0509, beta=0.9452
Estimated parameters: omega=0.0006337013643858419, alpha=0.0499, beta=0.9446
Estimated parameters: omega=0.01602850765117938, d=0.3177, beta=0.2779
Final log-likelihood = 1668.1033
Estimated parameters: omega=0.01589817231894883, d=0.2846, beta=0.2475
Final log-likelihood = 2673.3537


In [34]:
error_metrics_df_4, _, m_z_results_4, _ = strategy_4.in_sample_predictions()

In [35]:
error_metrics_df_4

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,1.295704,NaN,NaN,1.104215,NaN,NaN
BMSM OLS,4.284831,1.321581,0.906667,1.922571,1.438218,0.924628
GARCH,2.129224,NaN,NaN,1.320345,NaN,NaN
GARCH OLS,3.290568,0.452281,0.674409,1.725204,0.497548,0.690534
FIGARCH,1.980641,NaN,NaN,1.329444,NaN,NaN
FIGARCH OLS,3.605842,0.536656,0.704176,1.830976,0.473024,0.681841


In [36]:
m_z_results_4

{'BMSM': {'alpha_hat': 0.004860008629608473,
  'beta_hat': -1.0190876818351684,
  'alpha_p': 2.6744389473476468e-17,
  'beta_p': 8.933460641683627e-18},
 'GARCH': {'alpha_hat': 0.002765520225936744,
  'beta_hat': -0.12289330110466162,
  'alpha_p': 4.1015143749424175e-33,
  'beta_p': 8.315783186831612e-48},
 'FIGARCH': {'alpha_hat': 0.005353273006667357,
  'beta_hat': -1.0099316848503055,
  'alpha_p': 2.687394959651295e-28,
  'beta_p': 4.573434925262097e-34}}

In [37]:
strategy_4.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.498
Model:                            OLS   Adj. R-squared:                  0.495
Method:                 Least Squares   F-statistic:                     37.24
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           1.09e-28
Time:                        22:38:26   Log-Likelihood:                -44.177
No. Observations:                 843   AIC:                             98.35
Df Residuals:                     838   BIC:                             122.0
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1511      0.023     -6.625      0.000      -0.196      -0.106
x1             0.0923      0.044      2.116      0.034       0.007       0.178
x2            -0.0351      0.038     -0.932      0.351      -0.109       0.039
x3             0.2655      0.039      6.802      0.000       0.189       0.342
x4             0.0619      0.037      1.689      0.091      -0.010       0.134
==============================================================================
Omnibus:                       33.490   Durbin-Watson:                   0.034
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               22.444
Skew:                           0.277   Prob(JB):                     1.34e-05
Kurtosis:                       2.424   Cond. No.                         6.44
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [38]:
strategy_4.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.366
Model:                            OLS   Adj. R-squared:                  0.364
Method:                 Least Squares   F-statistic:                     28.53
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           1.43e-17
Time:                        22:38:31   Log-Likelihood:                -15.892
No. Observations:                 843   AIC:                             39.78
Df Residuals:                     839   BIC:                             58.73
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.3774      0.022    -17.203      0.000      -0.420      -0.334
x1            -0.0698      0.020     -3.488      0.000      -0.109      -0.031
x2             0.2970      0.035      8.417      0.000       0.228       0.366
x3             0.1079      0.032      3.364      0.001       0.045       0.171
==============================================================================
Omnibus:                       13.948   Durbin-Watson:                   0.058
Prob(Omnibus):                  0.001   Jarque-Bera (JB):                9.878
Skew:                           0.145   Prob(JB):                      0.00716
Kurtosis:                       2.556   Cond. No.                         4.84
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [39]:
strategy_4.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.441
Model:                            OLS   Adj. R-squared:                  0.439
Method:                 Least Squares   F-statistic:                     33.64
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           1.56e-20
Time:                        22:38:34   Log-Likelihood:                -138.49
No. Observations:                 843   AIC:                             285.0
Df Residuals:                     839   BIC:                             303.9
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.3534      0.025    -14.101      0.000      -0.402      -0.304
x1            -0.0703      0.021     -3.339      0.001      -0.112      -0.029
x2             0.2800      0.045      6.184      0.000       0.191       0.369
x3             0.0226      0.039      0.577      0.564      -0.054       0.099
==============================================================================
Omnibus:                       40.470   Durbin-Watson:                   0.095
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               34.609
Skew:                           0.423   Prob(JB):                     3.05e-08
Kurtosis:                       2.482   Cond. No.                         4.43
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""